In [38]:
from contextlib import asynccontextmanager
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

In [39]:
# Global model variable
model = None

In [40]:
@asynccontextmanager
async def lifespan(app: FastAPI):
    global model
    model = joblib.load("heart_disease_backend_model.joblib")
    yield

In [41]:
app = FastAPI(title="Heart Disease Predictor API", lifespan=lifespan)

In [42]:
# Define the expected JSON format from your frontend
class PatientData(BaseModel):
    age: int
    sex: int
    cp: int
    trestbps: float
    chol: float
    fbs: int
    restecg: int
    thalach: float
    exang: int
    oldpeak: float
    slope: int
    ca: int
    thal: int

In [43]:
@app.post("/predict")
def predict(data: PatientData):
    try:
        # Convert incoming JSON directly into a 1-row DataFrame
        input_df = pd.DataFrame([data.model_dump()])
        
        prediction = int(model.predict(input_df)[0])
        probability = float(model.predict_proba(input_df)[0][1])
        
        return {
            "prediction": prediction,
            "probability": round(probability, 4),
            "risk": "High Risk" if probability >= 0.50 else "Low Risk",
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [45]:
!uvicorn server:app --reload --port 8000

INFO:     Will watch for changes in these directories: ['C:\\Users\\Sujoy Kundu\\ML_Practical\\sklearn']
ERROR:    Error loading ASGI app. Could not import module "server".
